# Guided APT processing: from raw hits to a saved result

**Import → Inspect → Crop → Calibrate → Range → Reconstruct → Analyze → Export**

After a PyCCAPT update, choose **Kernel → Restart Kernel and Run All**. The first cell reports the loaded source and kernel; the panel must include **Crop** between Inspect and Calibrate. Then follow the tabs from left to right. You can start with **raw flight times and no mass column**. The original processing notebook remains available.

**Preview** lets you inspect a proposed change. **Accept** makes it the current result. Nothing is accepted automatically. If you change an earlier step, later results are cleared so they cannot refer to the wrong data. **Undo accepted change** restores the previous state.

[Plain-language workflow guide](../../../../docs/guided_workflow.md) · [Calibration equations](../../../../docs/calibration_math.md)


In [ ]:
from pathlib import Path
import sys

# Support running from a repository checkout as well as an editable install.
repository_root = None
for candidate_root in (Path.cwd(), *Path.cwd().parents):
    if (candidate_root / 'setup.py').is_file() and (candidate_root / 'pyccapt').is_dir():
        repository_root = candidate_root
        if str(candidate_root) not in sys.path:
            sys.path.insert(0, str(candidate_root))
        break

from pyccapt.calibration.tutorials.tutorials_helpers.helper_guided_startup import load_guided_workflow
show_guided_workflow = load_guided_workflow(repository_root=repository_root)


In [ ]:
guided_panel = show_guided_workflow()


## First use — raw data

1. **Import:** select your HDF5 file and leave HDF5 mode on **Auto detect**. Raw `/dld` hits and processed `/df` tables are supported. Detector channel streams must first be converted to hits in the raw-data analysis notebook.
2. **Inspect:** look at flight time and voltage versus event position and at the detector map. These plots work without mass values.
3. **Crop:** open the tab; an orange rectangle and cyan detector circle appear automatically. Drag the rectangle's left/right handles to select the smooth rise in DC voltage; its height does not filter voltage. Move the circle by its center and shrink/enlarge it with the edge/corner handles to exclude sparse edges and mouse bites. The starting circle encloses finite detector hits; it does not choose the scientific ROI for you. Optional numeric fields are under **Exact values and optional filters**. After typing a different interval, click **Update detector map**. Then click **Preview crop**, check the selected count and plots, and **Accept crop**. Drawing changes only the draft; editing a crop field clears an older preview. Start is included; stop is excluded. The full source stays in the session for reconstruction depth.
4. **Calibrate:** check the actual flight length (mm), time zero (ns) and pulse mode. Known saved metadata fills these fields when available; example values are not measurements. Tick the confirmation box, click **Preview physical mass**, check the result, then **Accept calibration**. This creates your initial mass spectrum. With processed data, you can instead preview and accept its existing mass.
5. **Range:** choose expected elements and charges, propose peaks, then review each identity and boundary. Accept, reject, defer, split or merge as needed. Proposals are suggestions, not chemical truth. Preview and accept the reviewed ranges.
6. **Reconstruct:** enter reviewed material/instrument values, or inspect the optional ICF/kf ratio panel. A detector hotspot is not an indexed crystal pole. If there is no reliable periodic signal, the panel explains what is missing instead of inventing a pair.
7. **Analyze:** inspect composition from accepted ranges. Unranged events, background and ambiguous molecular overlaps still need scientific review.
8. **Export:** save a **resumable session** at any time. Export accepted HDF5 to save the selected events, metadata and a partial NeXus mapping; its embedded session retains the full original source and crop. Files label unknown settings instead of filling them from unrelated defaults.

To refine calibration, the optional multireference panel needs independently known identities in clean, separated windows spanning the run. **Propose reference windows** uses your accepted initial mass spectrum by default; raw flight-time windows are also available. Fill in the known `mass_Da` identities and review both boundaries before confirming them. A candidate must pass separate validation and confirmation and must not worsen the available baseline reference metrics. If it fails, keep the existing calibration and inspect the reported reference/time bands. Better peak position does not necessarily mean narrower peaks.


## What is saved, and what still needs your judgement?

- The session saves your original source, stable event IDs, crop, accepted results, draft peak decisions, model settings and review history. Cropping does not overwrite the original file.
- The exported event table contains only the selected rows. Its metadata records the source and selected counts. Resume the embedded session to recover the full source; a cropped table alone is not a complete acquisition for reconstruction-depth calculations.
- A ratio **q = ICF/kf** does not uniquely determine both parameters. Useful inputs are the correct indexed pole and plane spacing, flight length, field, density, detection efficiency, complete acquisition history, and an independent ICF anchor. A fitted recipe that reproduces old coordinates does not prove those coordinates are physically correct.
- Peak candidates are ordered by mass proximity and simplicity, not probability. Locked decisions survive proposal reruns on the same accepted mass spectrum. Overlaps use first-row priority; edit or split them when possible.

Long calculations run in the background. **Cancel** discards unfinished results; a numerical kernel may finish its current call first. Let explicit file saves finish before closing the kernel. For programmatic access use `guided_panel.controller['workflow']`; do not modify its source table directly.
